# scPolASeq — PBMC 1k v3 Pipeline Notebook

End-to-end orchestration notebook for running the **scPolASeq** alternative polyadenylation (APA) pipeline on 10x Genomics PBMC 1k v3 data (human, GRCh38).

**Steps covered**
1. Import libraries & configure paths  
2. Download / verify human reference datasets on `/cluster` (GRCh38 genome, GENCODE GTF, PolyASite2 atlas)  
3. Download / verify PBMC 1k v3 FASTQs + 10x barcode whitelist  
4. Quality-check all required inputs  
5. Configure pipeline parameters  
6. Run Nextflow stub validation (fast, no containers)  
7. Run the full 9-stage pipeline  
8. Inspect results — PDUI matrix, APA events, model metrics  

> **Prerequisites**: QNAP NAS mounted at `/cluster` with ~15 GB free, Nextflow ≥ 24.04, Apptainer available in WSL.

## 1. Import Required Libraries

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from datetime import datetime

# ── Helper: run a shell command in WSL and stream output ─────────────────────
def wsl(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    """Run cmd inside WSL bash. Output is printed live; returns CompletedProcess."""
    result = subprocess.run(["wsl", "bash", "-lc", cmd], text=True,
                            capture_output=False)
    if check and result.returncode != 0:
        raise RuntimeError(f"WSL command failed (rc={result.returncode}):\n  {cmd}")
    return result

def wsl_out(cmd: str) -> str:
    """Run cmd in WSL and return stdout as a string (no live print)."""
    r = subprocess.run(["wsl", "bash", "-lc", cmd],
                       text=True, capture_output=True)
    return r.stdout.strip()

# ── Paths (WSL/Linux perspective — all ops go through wsl()) ─────────────────
CLUSTER         = "/cluster"
REF_DIR         = f"{CLUSTER}/datasets/reference/GRCh38"
POLYA_DIR       = f"{CLUSTER}/datasets/polya_atlas"
WL_DIR          = f"{CLUSTER}/datasets/10x_whitelists"
PBMC_FASTQ_DIR  = f"{CLUSTER}/datasets/pbmc1k/fastqs"
RESULTS_DIR     = f"{CLUSTER}/projects/scPolASeq_pbmc1k/results"
WORK_DIR        = f"{CLUSTER}/scratch/nextflow_work/scpolaseq_pbmc1k"

# Pipeline root in WSL path notation
PIPELINE_WSL = "/mnt/c/Users/megar/Documents/Megas/Hackaton/scPolASeq"

print(f"Pipeline dir : {PIPELINE_WSL}")
print(f"Cluster root : {CLUSTER}")
print(f"Host Python  : {sys.version.split()[0]}")
print(f"Date         : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()

# Quick WSL availability check
v = wsl_out("uname -r")
print(f"WSL kernel   : {v}")
nf = wsl_out("nextflow -version 2>/dev/null | head -1 || echo 'not found'")
print(f"Nextflow     : {nf}")


## 2. Load Human Reference Dataset

Download GRCh38 primary assembly (GENCODE 44), the GTF annotation, and the PolyASite 2.0 polyA atlas to `/cluster/datasets/`.  
Each download is skipped if the file already exists.

In [ ]:
# Verify /cluster is mounted and writable in WSL
r = subprocess.run(
    ["wsl", "bash", "-lc",
     "mountpoint -q /cluster && echo 'mounted' || echo 'NOT MOUNTED'; "
     "touch /cluster/.write_test && rm /cluster/.write_test && echo 'writable' || echo 'NOT WRITABLE'; "
     "df -h /cluster | tail -1"],
    text=True, capture_output=True
)
print(r.stdout)
if "NOT MOUNTED" in r.stdout:
    raise RuntimeError(
        "/cluster is not mounted in WSL.\n"
        "Fix: wsl -e bash -c 'sudo mount -t nfs 192.168.0.224:/cluster /cluster'"
    )
print("✓ /cluster is mounted and writable")


In [ ]:
GENOME_URL = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/GRCh38.primary_assembly.genome.fa.gz"
GTF_URL    = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.annotation.gtf.gz"
POLYA_URL  = "https://polyasite.unibas.ch/download/atlas/2.0/GRCh38.96/atlas.clusters.2.0.GRCh38.96.tsv.gz"

# Each download: skip if target exists and is non-empty, else curl | gunzip
def dl(url: str, dest: str, label: str):
    script = (
        f'if [ -s "{dest}" ]; then '
        f'  echo "[SKIP] {label} -> {dest}"; '
        f'else '
        f'  mkdir -p "$(dirname {dest})" && '
        f'  echo "Downloading {label}..." && '
        f'  curl --progress-bar -fL "{url}" | gunzip -c > "{dest}" && '
        f'  echo "✓ {label} ($(du -sh {dest} | cut -f1))"; '
        f'fi'
    )
    wsl(script)

dl(GENOME_URL, f"{REF_DIR}/genome.fa",               "GRCh38 genome  (~3 GB uncompressed)")
dl(GTF_URL,    f"{REF_DIR}/genes.gtf",               "GENCODE v44 GTF")
dl(POLYA_URL,  f"{POLYA_DIR}/polyasite2_GRCh38.tsv", "PolyASite 2.0 atlas")
print("\nReference download step complete.")


## 3. Data Preprocessing — PBMC 1k FASTQs + Barcode Whitelist

Download the 10x Genomics PBMC 1k v3 FASTQs (~5 GB) and the 10x v3 barcode whitelist.  
Also builds the `chrom.sizes` file from the genome index if `samtools` is available.

In [ ]:
PBMC_TAR_URL = "https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_fastqs.tar"
WL_URL       = "https://cf.10xgenomics.com/supp/cell-exp/whitelists/3M-february-2018.txt.gz"

# Barcode whitelist
wl_script = (
    f'if [ -s "{WL_DIR}/3M-february-2018.txt" ]; then '
    f'  echo "[SKIP] whitelist already present"; '
    f'else '
    f'  mkdir -p "{WL_DIR}" && '
    f'  echo "Downloading 10x v3 barcode whitelist..." && '
    f'  curl --progress-bar -fL "{WL_URL}" | gunzip -c > "{WL_DIR}/3M-february-2018.txt" && '
    f'  echo "✓ Whitelist: $(wc -l < {WL_DIR}/3M-february-2018.txt) barcodes"; '
    f'fi'
)
wsl(wl_script)

# PBMC 1k FASTQs
fastq_script = (
    f'if ls "{PBMC_FASTQ_DIR}/"*.fastq.gz 2>/dev/null | grep -q .; then '
    f'  echo "[SKIP] FASTQs already present: $(ls {PBMC_FASTQ_DIR}/*.fastq.gz | wc -l) files"; '
    f'else '
    f'  mkdir -p "{PBMC_FASTQ_DIR}" && '
    f'  echo "Downloading PBMC 1k v3 FASTQs (~5 GB)..." && '
    f'  curl --progress-bar -fL "{PBMC_TAR_URL}" '
    f'    | tar -xf - -C "{PBMC_FASTQ_DIR}" --strip-components=1 && '
    f'  echo "✓ FASTQs: $(ls {PBMC_FASTQ_DIR}/*.fastq.gz | wc -l) files"; '
    f'fi'
)
wsl(fastq_script)

print("\nAll input data ready.")


## 4. Quality Control — Input Readiness Check

Verify that every file expected by `conf/test_pbmc1k.config` is present and non-empty before Nextflow is invoked.

In [ ]:
checks = {
    "Genome FASTA":       f"{REF_DIR}/genome.fa",
    "GTF annotation":     f"{REF_DIR}/genes.gtf",
    "PolyASite atlas":    f"{POLYA_DIR}/polyasite2_GRCh38.tsv",
    "Barcode whitelist":  f"{WL_DIR}/3M-february-2018.txt",
    "PBMC L001 R1":       f"{PBMC_FASTQ_DIR}/pbmc_1k_v3_S1_L001_R1_001.fastq.gz",
    "PBMC L001 R2":       f"{PBMC_FASTQ_DIR}/pbmc_1k_v3_S1_L001_R2_001.fastq.gz",
    "PBMC L002 R1":       f"{PBMC_FASTQ_DIR}/pbmc_1k_v3_S1_L002_R1_001.fastq.gz",
    "PBMC L002 R2":       f"{PBMC_FASTQ_DIR}/pbmc_1k_v3_S1_L002_R2_001.fastq.gz",
    "Samplesheet":        f"{PIPELINE_WSL}/tests/data/samplesheet_pbmc1k.csv",
    "Test config":        f"{PIPELINE_WSL}/conf/test_pbmc1k.config",
    "main.nf":            f"{PIPELINE_WSL}/main.nf",
}

all_ok = True
print(f"{'Input':<24} {'Status':<10} {'Size':>10}")
print("─" * 48)
for label, path in checks.items():
    out = wsl_out(
        f'if [ -s "{path}" ]; then '
        f'  du -sh "{path}" | cut -f1; '
        f'else echo MISSING; fi'
    )
    if out == "MISSING":
        all_ok = False
        print(f"  {label:<22} {'✗ MISSING':<10}")
    else:
        print(f"  {label:<22} {'✓':<10} {out:>10}")

print()
if all_ok:
    print("✓ All inputs present — ready to run.")
else:
    print("✗ Some inputs missing — re-run download cells above.")


## 5. Configure Pipeline Parameters

Override any `test_pbmc1k.config` defaults here before launching Nextflow.  
Values are exported as environment variables consumed by `nextflow run`.

In [ ]:
CLUSTER_CONFIG = "/mnt/c/Users/megar/Documents/Megas/Hackaton/cluster_config/nextflow.config"
NXF_PROFILE    = "test_pbmc1k,local_qnap"
NXF_HOME       = "/cluster/scratch/.nextflow"
NXF_SIF_CACHE  = "/cluster/containers"

NXF_PARAMS = {
    "outdir":                            RESULTS_DIR,
    "apa_min_coverage":                  "5",
    "apa_min_pdui_delta":                "0.2",
    "apa_grouping_levels":               "cluster,cell_type",
    "apa_group_level":                   "cluster",
    "apa_model_type":                    "random_forest",
    "enable_single_cell_apa_projection": "false",
    "max_cpus":                          "8",
    "max_memory":                        "10 GB",
    "max_time":                          "12 h",
}

param_str = " ".join(f'--{k} "{v}"' for k, v in NXF_PARAMS.items())

def run_nextflow(extra_flags: str, label: str, work_subdir: str) -> bool:
    cmd = (
        f"export NXF_HOME={NXF_HOME} NXF_SINGULARITY_CACHEDIR={NXF_SIF_CACHE}; "
        f"cd {PIPELINE_WSL}; "
        f"nextflow run main.nf "
        f"  -profile {NXF_PROFILE} "
        f"  -c {CLUSTER_CONFIG} "
        f"  -work-dir {CLUSTER}/scratch/nextflow_work/{work_subdir} "
        f"  {extra_flags}"
    )
    print("=" * 68)
    print(f"  {label}")
    print("=" * 68)
    r = wsl(cmd, check=False)
    ok = r.returncode == 0
    print(f"\n{'✓ SUCCESS' if ok else '✗ FAILED'}  (exit code {r.returncode})")
    return ok

print("Nextflow runner configured.")
print(f"  Profile   : {NXF_PROFILE}")
print(f"  Work dir  : {CLUSTER}/scratch/nextflow_work/")
print(f"  Results   : {RESULTS_DIR}")


## 6. Stub Validation (Dimensionality Reduction Analogy)

Run Nextflow with `-stub-run` — all processes execute their `stub:` blocks (no containers, no real compute).  
This is equivalent to a "dry run" that validates channel wiring across all 9 stages without data.

In [ ]:
ok_stub = run_nextflow(
    extra_flags=(
        "-stub-run "
        f'--outdir "{CLUSTER}/projects/scPolASeq_stub/results" '
        '--max_cpus 4 --max_memory "4 GB"'
    ),
    label="Stub validation — 9-stage wiring check (no containers)",
    work_subdir="scpolaseq_stub",
)


## 7. Full Pipeline Run — PBMC 1k v3

Executes all 9 stages against the real PBMC 1k FASTQs using Apptainer containers (`local_qnap` profile).  
Estimated wall time: **2–4 h** on the notebook (8 CPUs, 10 GB RAM).

> Only run after the stub validation passed and all inputs are confirmed present.

In [ ]:
assert ok_stub, "⚠ Stub validation failed — fix errors above before proceeding."

ok_full = run_nextflow(
    extra_flags=param_str,
    label="Full pipeline — PBMC 1k v3 (9-stage APA + ML) — ~2-4 h",
    work_subdir="scpolaseq_pbmc1k",
)


## 8. Inspect Results — APA Events, PDUI Matrix, Model Metrics

Load the pipeline outputs from `/cluster/projects/scPolASeq_pbmc1k/results` and produce summary tables and plots.

In [ ]:
import glob
import pandas as pd

# Helper: find first matching file under RESULTS_DIR via WSL glob
def _find(pattern: str) -> str | None:
    out = wsl_out(f'find "{RESULTS_DIR}" -name "{pattern}" 2>/dev/null | head -1')
    return out if out else None

# Convert WSL path to Windows path for pandas to read
def _to_win(wsl_path: str) -> str:
    # /cluster/... is on NAS (NFS), but pandas is running on Windows.
    # We read via WSL cat → stdin.
    return wsl_path

def _load(pattern: str, label: str) -> "pd.DataFrame | None":
    import io
    wpath = _find(pattern)
    if not wpath:
        print(f"  [not found] {label}")
        return None
    content = wsl_out(f'cat "{wpath}"')
    df = pd.read_csv(io.StringIO(content), sep="\t")
    print(f"  ✓ {label}: {len(df):,} rows × {df.shape[1]} cols")
    return df

print(f"Reading results from: {RESULTS_DIR}\n")
df_events   = _load("apa_events.tsv",         "APA events")
df_pdui     = _load("pdui_usage_matrix.tsv",  "PDUI matrix")
df_scored   = _load("scored_apa_events.tsv",  "Scored events")
df_metrics  = _load("model_metrics.tsv",      "Model metrics")
df_features = _load("apa_features.tsv",       "Feature table")

if df_metrics is not None:
    print("\n── Model metrics ──────────────────────")
    print(df_metrics.to_string(index=False))

if df_events is not None and "is_significant" in df_events.columns:
    n_sig = int(df_events["is_significant"].sum())
    print(f"\n── APA events: {len(df_events):,} comparisons  |  {n_sig} significant ──")


In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle("scPolASeq — PBMC 1k v3  |  APA summary", fontsize=13)

    # 1. ΔPDUI distribution
    if df_events is not None and "delta_pdui" in df_events.columns:
        ax = axes[0]
        df_events["delta_pdui"].hist(bins=40, ax=ax, color="#4C72B0", edgecolor="white")
        ax.axvline(0.2,  color="red", linestyle="--", linewidth=1.2, label="|ΔPDUI|=0.2")
        ax.axvline(-0.2, color="red", linestyle="--", linewidth=1.2)
        ax.set_title("ΔPDUI Distribution"); ax.set_xlabel("ΔPDUI"); ax.set_ylabel("# comparisons")
        ax.legend(fontsize=8)

    # 2. APA score distribution
    if df_scored is not None and "apa_score" in df_scored.columns:
        ax = axes[1]
        df_scored["apa_score"].hist(bins=30, ax=ax, color="#55A868", edgecolor="white")
        ax.axvline(0.5, color="red", linestyle="--", linewidth=1.2, label="threshold=0.5")
        ax.set_title("ML APA Score Distribution"); ax.set_xlabel("APA score"); ax.legend(fontsize=8)

    # 3. Feature importances
    df_imp_path = _find("feature_importance.tsv")
    if df_imp_path:
        import io
        content = wsl_out(f'cat "{df_imp_path}"')
        df_imp = pd.read_csv(io.StringIO(content), sep="\t")
        ax = axes[2]
        df_imp.sort_values("importance").plot.barh(
            x="feature", y="importance", ax=ax, color="#C44E52", legend=False)
        ax.set_title("Feature Importance"); ax.set_xlabel("Importance")

    plt.tight_layout()
    plt.savefig("scpolaseq_pbmc1k_summary.png", dpi=120)
    plt.show()
    print("Figure saved → scpolaseq_pbmc1k_summary.png")

except ImportError:
    print("matplotlib not installed — pip install matplotlib")
